# Prétraitement du dataset Diabète – Sprint 1

Ce notebook couvre :
1. Lecture et inspection des données
2. Prétraitement initial (regroupement des classes)
3. Séparation train / validation / test
4. Normalisation des features

In [20]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

## Lecture du dataset
On charge le fichier CSV et on affiche un aperçu ainsi que les informations générales.

In [21]:
def read_data(file_path="../data/diabetes_binary_health_indicators_BRFSS2015.csv"):
    try:
        df = pd.read_csv(file_path)
        print("Aperçu des 5 premières lignes :")
        print(df.head())
        print("\nInformations générales :")
        print(df.info())
        print("\nValeurs manquantes par colonne :")
        print(df.isnull().sum())
        df.to_parquet("test_index_false.parquet", index=False)
        return df
    except FileNotFoundError:
        print(f"Erreur : le fichier {file_path} n'a pas été trouvé.")
        return None

df = read_data()

Aperçu des 5 premières lignes :
   Diabetes_binary  HighBP  HighChol  CholCheck   BMI  Smoker  Stroke  \
0              0.0     1.0       1.0        1.0  40.0     1.0     0.0   
1              0.0     0.0       0.0        0.0  25.0     1.0     0.0   
2              0.0     1.0       1.0        1.0  28.0     0.0     0.0   
3              0.0     1.0       0.0        1.0  27.0     0.0     0.0   
4              0.0     1.0       1.0        1.0  24.0     0.0     0.0   

   HeartDiseaseorAttack  PhysActivity  Fruits  ...  AnyHealthcare  \
0                   0.0           0.0     0.0  ...            1.0   
1                   0.0           1.0     0.0  ...            0.0   
2                   0.0           0.0     1.0  ...            1.0   
3                   0.0           1.0     1.0  ...            1.0   
4                   0.0           1.0     1.0  ...            1.0   

   NoDocbcCost  GenHlth  MentHlth  PhysHlth  DiffWalk  Sex   Age  Education  \
0          0.0      5.0      18.0  

## Prétraitement initial
- Regroupement des classes : 0 = no diabète ou prediabète, 1 = diabète
- Séparation des features (X) et de la cible (y)

In [22]:
def separate_data(df):
    X = df.drop("Diabetes_binary", axis=1)
    y = df["Diabetes_binary"]
    return X, y

if df is not None:
    X, y = separate_data(df)
    print("Distribution des classes :")
    print(y.value_counts())

Distribution des classes :
Diabetes_binary
0.0    218334
1.0     35346
Name: count, dtype: int64


## Séparation Train / Validation / Test
- 70% train
- 15% validation
- 15% test
- Stratification pour garder la proportion des classes

In [23]:
if df is not None:
    X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)
    X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5, random_state=42, stratify=y_temp)
    
    print("Taille train :", X_train.shape)
    print("Taille validation :", X_val.shape)
    print("Taille test :", X_test.shape)

Taille train : (177576, 21)
Taille validation : (38052, 21)
Taille test : (38052, 21)


## Normalisation des features
- Standardisation : moyenne=0, écart type=1
- Fit sur le train, transform sur validation et test

In [24]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)

print("Normalisation effectuée. Exemple :")
print(X_train_scaled[:5])

Normalisation effectuée. Exemple :
[[-0.86692187  1.16486098  0.19770711 -0.96495671  1.12096684 -0.20493045
  -0.32292458  0.56632089  0.75853319  0.48181784 -0.24224481  0.22728907
  -0.30393876 -0.47881617 -0.42993769 -0.48668106 -0.44917051 -0.88946385
   1.30094063  0.96354187 -0.02582259]
 [-0.86692187  1.16486098  0.19770711 -0.8136217   1.12096684 -0.20493045
  -0.32292458  0.56632089  0.75853319  0.48181784  4.12805538  0.22728907
  -0.30393876 -0.47881617 -0.42993769 -0.14262177  2.22632603 -0.88946385
  -0.33821355 -1.06446483  0.93882906]
 [-0.86692187 -0.85847154  0.19770711 -0.20828165 -0.89208705 -0.20493045
  -0.32292458  0.56632089 -1.31833387  0.48181784  4.12805538  0.22728907
  -0.30393876 -1.41464125 -0.42993769 -0.48668106 -0.44917051  1.12427278
  -0.01038271  0.96354187  0.45650324]
 [ 1.15350648 -0.85847154  0.19770711 -0.05694664  1.12096684 -0.20493045
  -0.32292458  0.56632089  0.75853319  0.48181784 -0.24224481  0.22728907
  -0.30393876 -0.47881617  0.51255